[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/07-integrations/02-sql_database_integration.ipynb)

In [1]:
# !pip install mbox pandas

# Loading and Resolving Against a SQL Database

Every notebook so far in this cookbook starts from a CSV. Real production data almost never does, it lives in a database. This notebook builds an M|BOX index directly from a real, publicly available SQL database, resolves a fuzzy mention against it, and writes the result back into the same database, the full round trip a production integration actually needs.

The database is [Chinook](https://github.com/lerocha/chinook-database), a well-known open-source sample database modeling a digital media store: customers, invoices, artists, albums, and tracks. It ships as a single `.sqlite` file, so this notebook needs no database server, just Python's built-in `sqlite3`.

In this notebook you will:

1. Fetch a real public SQLite database and load a table straight out of it with `pandas.read_sql_query`
2. Build an M|BOX index from that real data, genuine international names, accents included
3. Resolve a customer mentioned by a typo'd, unaccented name, and pull their real order history
4. Write the resolved match back into a new table in the same database with `to_sql`

> Note: this notebook downloads a ~1 MB public sample database on first run if it isn't already present in `datasets/`.

In [ ]:
import pandas as pd
import sqlite3

## 1. Get the database

`urlretrieve` only runs if the file isn't already sitting in `datasets/`, so re-running this cell is cheap and doesn't re-download anything.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

db_path = Path("datasets/chinook.sqlite")
if not db_path.exists():
    urlretrieve(
        "https://github.com/lerocha/chinook-database/raw/master/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite",
        db_path,
    )
print(f"{db_path} ({db_path.stat().st_size:,} bytes)")

datasets\chinook.sqlite (1,011,712 bytes)


## 2. Load a real table

A plain `sqlite3.connect()` and `pandas.read_sql_query()`, no ORM, no ceremony. This same pattern works unchanged against Postgres, MySQL, or SQL Server, swap the connection object for a SQLAlchemy engine and everything downstream is identical.

In [ ]:
conn = sqlite3.connect("datasets/chinook.sqlite")
customers = pd.read_sql_query(
    "SELECT CustomerId, FirstName, LastName, Company, City, Country, Email FROM Customer", conn
)
customers["full_name"] = customers["FirstName"] + " " + customers["LastName"]
print(f"{len(customers)} real customer records")
customers.head()

59 real customer records


,CustomerId,FirstName,LastName,Company,City,Country,Email,full_name
0,1,Luís,Gonçalves,Embraer - Empresa Brasileira de Aeronáutica S.A.,São José dos Campos,Brazil,luisg@embraer.com.br,Luís Gonçalves
1,2,Leonie,Köhler,NaN,Stuttgart,Germany,leonekohler@surfeu.de,Leonie Köhler
2,3,François,Tremblay,NaN,Montréal,Canada,ftremblay@gmail.com,François Tremblay
3,4,Bjørn,Hansen,NaN,Oslo,Norway,bjorn.hansen@yahoo.no,Bjørn Hansen
4,5,František,Wichterlová,JetBrains s.r.o.,Prague,Czech Republic,frantisekw@jetbrains.com,František Wichterlová


These are genuine international names, diacritics included, `František Wichterlová`, `Luís Gonçalves`, straight out of the actual sample database, not written for this notebook.

## 3. Build the index and resolve a real mention

A support agent, or an upstream system, hands you `"luiz goncalves"`, no accents, one letter off. Nothing about this query was prepared for the demo, it's exactly the kind of input a real lookup field receives.

In [ ]:
from mbox.indexing import TableIndexer
from mbox.recall import TableRecallConfig, TableRecallFieldConfig, TableRecallMode

index = TableIndexer.create_index(customers, index_columns=["full_name"], tmp_dir="tmp_index_customers")
config = TableRecallConfig(
    fields=[TableRecallFieldConfig(input_column="full_name", indexed_column="full_name",
                                    minimum_quality=0, weight=100, mode=TableRecallMode.APPROX)],
    max_results=3, min_total_match_value=0, include_field_scores=True
)

query = "luiz goncalves"
resolved = index.match(queries=pd.DataFrame({"full_name": [query]}), config=config)
resolved[["full_name_candidate", "full_name_score"]]

mbpie: 33 modules, 566 methods, 8 classes, 18 enums
  args: 426 required, 254 optional, 37 keywords, 39 flags, 26 arrays
  types: 372 int, 337 str, 1 double, 72 object


,full_name_candidate,full_name_score
0,Luís Gonçalves,81


A missing accent and a typo'd first name still resolve cleanly. Notice this needed no special accent-handling configuration, `CharacterMapping` from `02-data-harmonization/` exists for cases that need explicit control over normalization, but plain `APPROX` already tolerates missing diacritics on its own here.

In [ ]:
matched_row = customers.iloc[int(resolved.iloc[0]["index_row"])]
print(matched_row[["CustomerId", "full_name", "Company", "City", "Country", "Email"]])

orders = pd.read_sql_query(
    f"SELECT InvoiceId, InvoiceDate, Total FROM Invoice WHERE CustomerId = {int(matched_row['CustomerId'])}", conn
)
print(f"\n{len(orders)} real invoices on file:")
orders

CustomerId                                                   1
full_name                                       Luís Gonçalves
Company       Embraer - Empresa Brasileira de Aeronáutica S.A.
City                                       São José dos Campos
Country                                                 Brazil
Email                                     luisg@embraer.com.br
Name: 0, dtype: object

7 real invoices on file:


,InvoiceId,InvoiceDate,Total
0,98,2022-03-11 00:00:00,3.98
1,121,2022-06-13 00:00:00,3.96
2,143,2022-09-15 00:00:00,5.94
3,195,2023-05-06 00:00:00,0.99
4,316,2024-10-27 00:00:00,1.98
5,327,2024-12-07 00:00:00,13.86
6,382,2025-08-07 00:00:00,8.91


One resolved match, joined straight into that customer's real order history from a second table, the payoff of doing the resolution against the actual database instead of a disconnected export.

## 4. Write the result back

The other half of a real integration: persist the resolution, not just print it. A new table, `ResolvedLookups`, written with `to_sql`, right back into the same database file.

In [ ]:
result = pd.DataFrame([{
    "query_text": query,
    "resolved_customer_id": int(matched_row["CustomerId"]),
    "resolved_name": matched_row["full_name"],
    "match_confidence": int(resolved.iloc[0]["full_name_score"]),
}])
result.to_sql("ResolvedLookups", conn, if_exists="replace", index=False)

pd.read_sql_query("SELECT * FROM ResolvedLookups", conn)

,query_text,resolved_customer_id,resolved_name,match_confidence
0,luiz goncalves,1,Luís Gonçalves,81


That last read is proof, not decoration, it queried the table back out of the database file on disk, the same way any other system reading from Chinook would see it.

## 5. Practical notes

**Build the index from a query result, not from the whole table.** `SELECT` only the columns you actually index and match on. A wide production table often carries far more columns than any single M|BOX field configuration needs, and pulling all of them into the DataFrame just to index three costs memory for nothing.

**`read_sql_query` takes a `chunksize` for tables too large to load at once.** Iterate over chunks, match each one, and either `to_sql(..., if_exists="append")` the results or accumulate them, the same batch-matching pattern `06-agentic-ai/08` used on an in-memory DataFrame works identically on chunks pulled straight from a database cursor.

**This isn't SQLite-specific.** Everything here, `read_sql_query`, `to_sql`, the M|BOX calls in between, works unchanged against a `sqlalchemy.create_engine(...)` connection to Postgres, MySQL, or SQL Server. The database engine only shows up in one line, the connection string.

**Don't rebuild the index per query.** Build it once from the table snapshot, reuse it for every resolution in the batch or session, and rebuild only when the underlying table has actually changed, the same reuse discipline as everywhere else in this cookbook.

## Next steps

- **`01-fastapi_match_service.ipynb`** - expose a resolution like this one as an HTTP endpoint other services can call